# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# TODO: Load environment variables
# load_dotenv()

load_dotenv('.env')

True

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
# chroma_client = chromadb.PersistentClient(path="chromadb")
chroma_client = chromadb.PersistentClient(path="chromadb")

In [9]:
chroma_client.__getstate__()

{'_identifier': 'chromadb',
 'tenant': 'default_tenant',
 'database': 'default_database',
 '_server': <chromadb.api.rust.RustBindingsAPI at 0x10f678dd0>,
 '_admin_client': <chromadb.api.client.AdminClient at 0x1082729f0>}

### Collection

In [10]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
# embedding_fn = embedding_functions.OpenAIEmbeddingFunction()
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY")
    )

In [ ]:
#chroma_client.delete_collection(name="udaplay")

In [12]:
# TODO: Create a collection
# Choose any name you want
# collection = chroma_client.create_collection(
#    name="udaplay",
#    embedding_function=embedding_fn
#)

collection = chroma_client.create_collection(
   name="udaplay",
   embedding_function=embedding_fn
)

In [15]:
collection.__getstate__()

{'_client': <chromadb.api.rust.RustBindingsAPI at 0x10f678dd0>,
 '_model': Collection(id=UUID('28e068c0-4714-472a-99b6-871d54ad85b7'), name='udaplay', configuration_json={'hnsw': {'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': {'type': 'known', 'name': 'openai', 'config': {'api_base': None, 'api_key_env_var': 'OPENAI_API_KEY', 'api_type': None, 'api_version': None, 'default_headers': None, 'deployment_id': None, 'dimensions': None, 'model_name': 'text-embedding-ada-002', 'organization_id': None}}}, serialized_schema={'defaults': {'string': {'fts_index': {'enabled': False, 'config': {}}, 'string_inverted_index': {'enabled': True, 'config': {}}}, 'float_list': {'vector_index': {'enabled': False, 'config': {'space': 'cosine', 'embedding_function': {'type': 'known', 'name': 'openai', 'config': {'api_base': None, 'api_key_env_var': 'OPENAI_API_KEY', 'api_type': None, 'api_v

### Add documents

In [16]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
    

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]
    print("id", doc_id)
    print("Content: ", content)

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

id 001
Content:  [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
id 002
Content:  [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.
id 003
Content:  [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
id 004
Content:  [PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.
id 005
Content:  [PlayStation 5] Marvel's Spider-Man 2 (2023) - The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.
id 006
Content:  [Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regi

In [18]:
# Example Query
query=["is Super Mario available for Nitendo 64?"]

results = collection.query(
    query_texts=query,
    n_results=2,
)

results

{'ids': [['009', '008']],
 'embeddings': None,
 'documents': [["[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
   '[Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'Publisher': 'Nintendo',
    'Platform': 'Nintendo 64',
    'YearOfRelease': 1996,
    'Genre': 'Platformer',
    'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
    'Name': 'Super Mario 64'},
   {'Publisher': 'Nintendo',
    'Platform': 'Super Nintendo Entertainment System (SNES)',
    'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur L